# 04 — Evaluation: CATE MAE, top-k decision quality, slice-level breakdown

**Project:** Training ROI Predictor (H08)

Three release deliverables:
1. CATE MAE vs ground truth — the truth-anchored bias check.
2. Top-k decision quality (uplift@k) — does targeting the top-k by predicted CATE actually capture the highest-uplift employees?
3. Per-dept × per-training-id breakdown — which intervention to recommend in which dept.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from training_roi.data import PROCESSED, load_panel, make_training_artifacts
from training_roi import models
from training_roi.models import fit_x_learner

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

In [ ]:
PARQUET = PROCESSED / 'training_outcomes.parquet'
df = load_panel() if PARQUET.exists() else make_training_artifacts()
df_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['treatment'])
try:
    xl = models.load('x_learner.joblib')
except FileNotFoundError:
    xl = fit_x_learner(df_train); models.save(xl)

## 1. Headline scorecard

In [ ]:
cate_test = xl.predict_cate(df_test)
scorecard = pd.Series({
    'mean_pred_CATE': cate_test.mean(),
    'mean_true_CATE': df_test['true_cate'].mean(),
    'MAE_vs_truth': mean_absolute_error(df_test['true_cate'], cate_test),
    'spearman_vs_truth': pd.Series(cate_test).corr(df_test['true_cate'].reset_index(drop=True), method='spearman'),
}).round(3)
scorecard.to_frame('value')

## 2. Uplift @ top-k decision quality

Pick the top-k by predicted CATE; what fraction of those have the largest *true* CATE?

In [ ]:
df_t = df_test.assign(pred=cate_test).reset_index(drop=True)
rows = []
for k_pct in [0.05, 0.10, 0.20, 0.50]:
    k = int(np.ceil(len(df_t) * k_pct))
    pred_top = df_t.sort_values('pred', ascending=False).head(k)
    true_top_set = set(df_t.sort_values('true_cate', ascending=False).head(k).index)
    overlap = len(set(pred_top.index) & true_top_set) / k
    rows.append({'k_pct': k_pct, 'overlap': overlap, 'k': k})
uplift_table = pd.DataFrame(rows).round(3)
uplift_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(uplift_table['k_pct'], uplift_table['overlap'], marker='o', color='#3a7ca5')
ax.axhline(0.7, color='red', ls='--', label='target 0.7')
ax.set_xlabel('top-k fraction'); ax.set_ylabel('overlap with true top-k')
ax.set_title('Decision quality at top-k')
ax.legend()
plt.tight_layout(); plt.show()

## 3. Population uplift histogram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
sns.histplot(cate_test, bins=40, ax=ax, color='#9c6644')
ax.axvline(cate_test.mean(), color='red', ls='--', label=f'mean={cate_test.mean():.2f}')
ax.set_title('Predicted population uplift distribution')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Per-dept × per-training breakdown

In [ ]:
per_slice = (
    df_t.groupby(['dept', 'training_id'])['pred']
    .mean().unstack().round(3)
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(per_slice, cmap='RdYlGn', center=0.5, annot=True, fmt='.2f', ax=ax)
ax.set_title('Mean predicted CATE by dept × training_id')
plt.tight_layout(); plt.show()

## 5. SUTVA / ignorability sanity checks (qualitative)

We can't test SUTVA on a synthetic generator definitively, but we can confirm: (a) propensities are not at the boundary (no perfect-confounder), and (b) the residual outcome distribution does not have suspicious bimodality.

In [ ]:
X = xl.preproc.transform(df_train[xl.feature_cols])
props = xl.propensity.predict_proba(X)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(props, bins=30, color='#3a7ca5', edgecolor='white')
axes[0].axvline(0.05, color='red', ls='--'); axes[0].axvline(0.95, color='red', ls='--')
axes[0].set_title('Propensity distribution (clipped at [0.05, 0.95])')
resid = df_train['perf_uplift'].values
axes[1].hist(resid, bins=30, color='#4a7c59', edgecolor='white')
axes[1].set_title('Observed perf_uplift residual')
plt.tight_layout(); plt.show()

## 6. Bootstrap CI coverage

In [ ]:
sample = df_test.sample(60, random_state=0)
point, lo, hi = xl.predict_cate_with_ci(sample, df_train, n_boot=20, alpha=0.10, seed=1)
covered = ((sample['true_cate'].values >= lo) & (sample['true_cate'].values <= hi)).mean()
print(f'90% bootstrap CI empirical coverage: {covered:.2%}')

## 7. Release-readiness checklist

| Gate | Target | Result |
|---|---|---|
| MAE vs ground-truth CATE | ≤ 0.30 | see scorecard |
| Top-10% decision overlap | ≥ 0.50 | see §2 |
| Bootstrap CI coverage | within ±10pp of 90% | see §6 |

Documented next step: replace logistic propensity with a calibrated GBM; widen the hyperparameter search; estimate ATT (average treatment-effect on treated) as well as ATE (`docs/04_evaluation.md`).